In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
import gradio as gr
import os

class F1PredictionSystem:
    def __init__(self):
        self.lstm_model = None
        self.scalers = {}
        self.encoders = {}
        self.active_drivers = {}
        self.all_drivers_data = {}
        self.circuit_names = []
        
    def load_and_merge_data(self):
        """載入並合併所有數據"""
        print("載入數據檔案...")
        
        # 載入所有CSV檔案
        try:
            drivers = pd.read_csv('./data/drivers_updated.csv')
            laps = pd.read_csv('./data/fastest_laps_updated.csv')
            teams = pd.read_csv('./data/teams_updated.csv')
            
            # 標準化欄位名稱
            drivers = drivers.rename(columns={'Car': 'Team'})
            laps = laps.rename(columns={'Car': 'Team'})
            
            print(f"載入: 車手({drivers.shape[0]}) 圈速({laps.shape[0]}) 車隊({teams.shape[0]})")
            
            # 合併數據
            merged = pd.merge(drivers, laps, on=['Driver', 'Team', 'year'], how='left', suffixes=('', '_lap'))
            merged = pd.merge(merged, teams, on=['Team', 'year'], how='left', suffixes=('', '_team'))
            
            print(f"合併後數據: {merged.shape}")
            return merged
            
        except Exception as e:
            print(f"載入數據失敗: {e}")
            return self._create_sample_data()
    
    def _create_sample_data(self):
        """創建示例數據"""
        np.random.seed(42)
        drivers = ['Hamilton', 'Verstappen', 'Leclerc', 'Russell', 'Sainz', 'Norris']
        teams = ['Mercedes', 'Red Bull', 'Ferrari', 'McLaren']
        circuits = ['Monaco', 'Silverstone', 'Monza', 'Spa']
        years = range(2020, 2025)
        
        data = []
        for year in years:
            for circuit in circuits:
                for i, driver in enumerate(drivers):
                    data.append({
                        'Driver': driver,
                        'Team': teams[i % len(teams)],
                        'Grand Prix': circuit,
                        'year': year,
                        'Pos': np.random.randint(1, 21),
                        'PTS': np.random.uniform(0, 25)
                    })
        return pd.DataFrame(data)
    
    def prepare_data(self, df):
        """準備訓練數據"""
        print("準備數據...")
        
        # 填充缺失值
        df = df.fillna({'Driver': 'Unknown', 'Team': 'Unknown', 'Grand Prix': 'Unknown'})
        
        # 存儲車手數據
        self._store_driver_data(df)
        
        # 創建編碼器
        self.encoders['driver'] = LabelEncoder().fit(df['Driver'])
        self.encoders['team'] = LabelEncoder().fit(df['Team'])
        self.encoders['circuit'] = LabelEncoder().fit(df['Grand Prix'])
        
        self.circuit_names = sorted(df['Grand Prix'].unique())
        
        # 編碼特徵
        df['driver_encoded'] = self.encoders['driver'].transform(df['Driver'])
        df['team_encoded'] = self.encoders['team'].transform(df['Team'])
        df['circuit_encoded'] = self.encoders['circuit'].transform(df['Grand Prix'])
        
        # 計算實力特徵
        return self._calculate_skills(df)
    
    def _store_driver_data(self, df):
        """存儲車手完整數據"""
        max_year = df['year'].max()
        
        for driver in df['Driver'].unique():
            driver_data = df[df['Driver'] == driver]
            last_year = driver_data['year'].max()
            
            # 只保留近期活躍的車手
            if max_year - last_year <= 5:
                current_team = driver_data[driver_data['year'] == last_year]['Team'].iloc[0]
                self.active_drivers[driver] = {
                    'current_team': current_team,
                    'last_year': last_year,
                    'total_races': len(driver_data)
                }
                self.all_drivers_data[driver] = driver_data
    
    def _calculate_skills(self, df):
        """計算車手和車隊實力"""
        print("計算實力特徵...")
        
        # 位置分數計算
        def position_score(pos):
            if pd.isna(pos):
                return 0
            try:
                pos = float(pos)
                if pos == 1: return 100
                elif pos <= 3: return 85 - (pos-1) * 7.5
                elif pos <= 10: return 70 - (pos-3) * 5
                else: return max(0, 20 - pos)
            except:
                return 0
        
        df['position_score'] = df['Pos'].apply(position_score)
        
        # 車手實力 - 使用歷史平均表現
        driver_skills = df.groupby('Driver')['position_score'].mean()
        team_skills = df.groupby('Team')['position_score'].mean()
        
        df['driver_skill'] = df['Driver'].map(driver_skills)
        df['team_skill'] = df['Team'].map(team_skills)
        
        # 標準化到0-100範圍
        scaler = MinMaxScaler(feature_range=(0, 100))
        df['driver_skill'] = scaler.fit_transform(df[['driver_skill']]).flatten()
        df['team_skill'] = scaler.fit_transform(df[['team_skill']]).flatten()
        
        return df
    
    def create_sequences(self, df, seq_length=5):
        """創建LSTM序列數據"""
        print(f"創建序列數據，長度: {seq_length}")
        
        sequences, targets = [], []
        
        for driver in df['Driver'].unique():
            driver_data = df[df['Driver'] == driver].sort_values('year')
            
            if len(driver_data) < seq_length + 1:
                continue
                
            for i in range(len(driver_data) - seq_length):
                seq_data = driver_data.iloc[i:i+seq_length]
                target_data = driver_data.iloc[i+seq_length]
                
                # 序列特徵
                seq_features = []
                for _, row in seq_data.iterrows():
                    seq_features.append([
                        row['driver_encoded'], row['team_encoded'], 
                        row['circuit_encoded'], row['year'],
                        row['driver_skill'], row['team_skill']
                    ])
                
                sequences.append(seq_features)
                targets.append(target_data['position_score'] / 100.0)  # 標準化目標
        
        return np.array(sequences), np.array(targets)
    
    def build_model(self, seq_length, feature_dim):
        """建立LSTM模型"""
        model = tf.keras.Sequential([
            tf.keras.layers.LSTM(64, return_sequences=True, input_shape=(seq_length, feature_dim)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.LSTM(32),
            tf.keras.layers.Dense(16, activation='relu'),
            tf.keras.layers.Dense(1, activation='sigmoid')
        ])
        
        model.compile(optimizer='adam', loss='mse', metrics=['mae'])
        return model
    
    def train(self, df):
        """訓練模型"""
        print("開始訓練...")
        
        # 準備數據
        df_processed = self.prepare_data(df)
        X_seq, y_seq = self.create_sequences(df_processed)
        
        if len(X_seq) == 0:
            raise ValueError("序列數據不足")
        
        # 標準化序列數據
        X_reshaped = X_seq.reshape(-1, X_seq.shape[-1])
        self.scalers['sequence'] = StandardScaler()
        X_scaled = self.scalers['sequence'].fit_transform(X_reshaped)
        X_scaled = X_scaled.reshape(X_seq.shape)
        
        # 訓練模型
        self.lstm_model = self.build_model(X_seq.shape[1], X_seq.shape[2])
        
        # 分割數據
        split_idx = int(0.8 * len(X_scaled))
        X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]
        y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]
        
        # 訓練
        self.lstm_model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=50, batch_size=32, verbose=1,
            callbacks=[tf.keras.callbacks.EarlyStopping(patience=10)]
        )
        
        self.df_processed = df_processed
        print("訓練完成！")
    
    def predict_driver_performance(self, driver, circuit, year):
        """預測車手表現"""
        if driver not in self.active_drivers:
            return None
            
        try:
            current_team = self.active_drivers[driver]['current_team']
            
            # 編碼特徵
            driver_enc = self.encoders['driver'].transform([driver])[0]
            team_enc = self.encoders['team'].transform([current_team])[0]
            circuit_enc = self.encoders['circuit'].transform([circuit])[0]
            
            # 創建序列（使用歷史數據或平均值）
            sequence = []
            for i in range(5):  # 序列長度
                hist_year = year - 5 + i + 1
                
                # 嘗試獲取歷史數據
                hist_data = self.df_processed[
                    (self.df_processed['Driver'] == driver) & 
                    (self.df_processed['year'] == hist_year)
                ]
                
                if not hist_data.empty:
                    driver_skill = hist_data['driver_skill'].mean()
                    team_skill = hist_data['team_skill'].mean()
                else:
                    # 使用車手平均實力
                    driver_skill = self.df_processed[
                        self.df_processed['Driver'] == driver
                    ]['driver_skill'].mean()
                    team_skill = self.df_processed[
                        self.df_processed['Team'] == current_team
                    ]['team_skill'].mean()
                
                sequence.append([driver_enc, team_enc, circuit_enc, hist_year, driver_skill, team_skill])
            
            # 預測
            seq_array = np.array([sequence])
            seq_reshaped = seq_array.reshape(-1, seq_array.shape[-1])
            seq_scaled = self.scalers['sequence'].transform(seq_reshaped)
            seq_scaled = seq_scaled.reshape(seq_array.shape)
            
            performance = self.lstm_model.predict(seq_scaled, verbose=0)[0][0] * 100
            
            # 賽道調整
            circuit_factor = self._get_circuit_factor(driver, circuit)
            adjusted_performance = performance * circuit_factor
            
            return {
                'driver': driver,
                'team': current_team,
                'performance': adjusted_performance,
                'circuit_factor': circuit_factor
            }
            
        except Exception as e:
            print(f"預測 {driver} 時出錯: {e}")
            return None
    
    def _get_circuit_factor(self, driver, circuit):
        """賽道適應性係數"""
        # 簡化的賽道係數
        circuit_factors = {
            'Monaco': 1.1, 'Singapore': 1.08, 'Hungary': 1.05,
            'Italy': 0.95, 'Belgium': 1.03, 'Monza': 0.95
        }
        base_factor = circuit_factors.get(circuit, 1.0)
        
        # 添加車手特異性（基於哈希值的穩定隨機性）
        driver_hash = hash(driver) % 100
        adjustment = (driver_hash - 50) / 1000  # -0.05 到 +0.05
        
        return max(0.9, min(1.1, base_factor + adjustment))
    
    def predict_race(self, circuit, year, selected_drivers, top_n=10):
        """預測比賽結果"""
        print(f"預測 {circuit} {year} 年比賽")
        
        results = []
        for driver in selected_drivers:
            result = self.predict_driver_performance(driver, circuit, year)
            if result:
                # 添加比賽日隨機性
                race_day_factor = np.random.normal(1.0, 0.08)
                result['final_performance'] = result['performance'] * race_day_factor
                results.append(result)
        
        # 排序並分配排名
        results.sort(key=lambda x: x['final_performance'], reverse=True)
        for i, result in enumerate(results[:top_n]):
            result['position'] = i + 1
            
        return results


def create_interface(prediction_system):
    """創建Gradio界面"""
    
    def format_results(circuit, year, drivers, top_n):
        if not drivers:
            return "請選擇車手"
            
        try:
            results = prediction_system.predict_race(circuit, int(year), drivers, int(top_n))
            
            if not results:
                return "預測失敗"
            
            output = f"## 🏁 {circuit} {year} 年預測結果\n\n"
            output += "| 排名 | 車手 | 車隊 | 表現分數 | 賽道係數 | 最終表現 |\n"
            output += "|------|------|------|----------|----------|----------|\n"
            
            for r in results:
                output += f"| {r['position']} | {r['driver']} | {r['team']} | "
                output += f"{r['performance']:.1f} | {r['circuit_factor']:.3f} | {r['final_performance']:.1f} |\n"
            
            return output
            
        except Exception as e:
            return f"預測錯誤: {str(e)}"
    
    def update_drivers(year):
        """更新年份對應的活躍車手"""
        try:
            year_int = int(year)
            # 簡化：直接返回所有活躍車手
            active_list = list(prediction_system.active_drivers.keys())
            active_list.sort()
            return gr.Dropdown(choices=active_list, value=active_list[:10])
        except:
            return gr.Dropdown(choices=[], value=[])
    
    # 初始化
    active_drivers = list(prediction_system.active_drivers.keys())
    active_drivers.sort()
    
    with gr.Blocks(title="F1 預測系統", theme=gr.themes.Soft()) as interface:
        gr.Markdown("# 🏎️ F1 比賽預測系統（精簡版）")
        
        with gr.Row():
            with gr.Column():
                circuit = gr.Dropdown(
                    choices=prediction_system.circuit_names,
                    label="賽道",
                    value=prediction_system.circuit_names[0] if prediction_system.circuit_names else None
                )
                
                year = gr.Slider(1980, 2030, value=2024, step=1, label="年份")
                
                drivers = gr.Dropdown(
                    choices=active_drivers,
                    label="選擇車手",
                    multiselect=True,
                    value=active_drivers[:10]
                )
                
                top_n = gr.Slider(3, 20, value=10, step=1, label="顯示前N名")
                
                predict_btn = gr.Button("🚀 開始預測", variant="primary")
            
            with gr.Column():
                results = gr.Markdown("選擇參數後點擊預測")
        
        # 事件綁定
        predict_btn.click(
            fn=format_results,
            inputs=[circuit, year, drivers, top_n],
            outputs=[results]
        )
        
        year.change(
            fn=update_drivers,
            inputs=[year],
            outputs=[drivers]
        )
    
    return interface


def main():
    """主函數"""
    print("🏎️ F1 預測系統啟動...")
    
    # 初始化系統
    system = F1PredictionSystem()
    
    # 載入並訓練
    df = system.load_and_merge_data()
    system.train(df)
    
    # 啟動界面
    interface = create_interface(system)
    print("🚀 啟動 Gradio 界面...")
    interface.launch()


if __name__ == '__main__':
    main()

ModuleNotFoundError: No module named 'tensorflow'